# MNIST learnable reduction, take 2 — with a `tau` that is not 10× hot

`mnist_reduction_kaggle.ipynb` trained 7 tapers and they lost up to 7 pp against `1x2000`. **That
result does not mean what it looks like.** This notebook re-runs the same models with one thing
changed, and it is not the architecture.

## What went wrong

`GroupSum.forward` ends in `x.sum(dim=-1) / self.tau`. So with `tau` held constant, **the logit
range is set by the group size**, and the group size is exactly the thing the reduction grid
sweeps. Tapering from 2000 to 100 does not only shrink the popcount — it shrinks the logit range
from 60 to 3, and puts a **hard floor of 0.370 under cross-entropy** that no amount of learning
can get below.

| final width | group | logit range at τ=3.3333 | CE floor | observed final loss |
|---|---|---|---|---|
| 100 | 10 | 3.0 | **0.370** | 0.58 – 0.63 |
| 200 | 20 | 6.0 | 0.022 | 0.122 |
| 500 | 50 | 15.0 | ~0 | 0.024 |
| 2000 | 200 | 60.0 | ~0 | 0.0008 |

Every taper ending at 100 sat jammed against that floor for all 30 epochs. And the floor is not
just cosmetic: a correctly classified easy example still returns a gradient of magnitude ≥ 0.31
forever, so it competes for the update budget with genuinely hard ones. The optimizer never gets
to concentrate.

**The proof that this is about group size and not about tapering:** `1x100` shows the identical
signature — group 10, loss stuck at 0.79, 88.37% — with no taper anywhere in it.

⚠️ So the previous notebook's headline is **retracted**: it measured `tau`, not the taper. Its
one trustworthy row is `3x[2000, 300, 500]` at 97.56%, which was never floor-limited.

## What this notebook holds fixed, and why that is the hard part

The previous grid froze `tau` *specifically* to avoid confounding — the JSC study had already
lost a run to a `tau` error that read as an architectural finding. Freezing it was the wrong fix:

> **When a hyperparameter's correct value is a function of the swept axis, holding it constant is
> not a control. It is a systematic bias against one end of the sweep.**

The control is to scale it by the known relationship. The problem is that MNIST gives us **one
anchor** — upstream's `examples/mnist.py` uses `tau = 1/0.3` on a 1000-wide final layer, group
100, range 30 — and one anchor cannot fix an exponent. So both defensible readings get measured
instead of assumed:

- **power law** — `tau ∝ width**0.57`, the exponent JSC's four anchors actually measure
  (`dse/grid.py: tau_for`). Lets the logit range grow with width, sublinearly.
- **flat range** — `tau ∝ width`, holding the range at the anchor's 30 for every config.

Both reproduce `tau = 3.3333` at width 1000 exactly, by construction.

## The three groups

| | what it re-runs | why |
|---|---|---|
| **A-reduction** (7) | the whole reduction grid at power-law `tau` | paired against 7 trained numbers; only `tau` differs |
| **B-ladder** (5) | `1x100/200/300/500/2000` at power-law `tau` | **the baselines are confounded too** — a taper at group 10 must be read against a single layer at group 10, and that rung was also 10× hot. `1x1000` is the anchor and is deliberately absent. |
| **C-schedule** (2) | `1x100` and `2x[2000,100]` at flat-range `tau` | gives **three tau points on one architecture**, so the schedule is measured rather than picked |

Nothing else moves: same seed, same 30 epochs, same LR schedule, same binarization, same
`train_one`. Every slug carries its `tau` so the old checkpoints are never overwritten.

## Before you run anything

**Accelerator → GPU**, **Internet → On**. ~70 minutes for all 14. To continue an unfinished run,
add the previous run's Output as an input dataset.

## What you get out

`<slug>_checkpoint.pt` and `<slug>_testvectors.npz` per config, in the format
`exporter/extract.py` reads. Download into `training/artifacts/mnist/`.


In [ ]:
# ---- environment check: fail loudly and early ----
import subprocess, sys, torch

print('torch     :', torch.__version__)
print('cuda avail:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu       :', torch.cuda.get_device_name(0))
    print('cuda (torch built against):', torch.version.cuda)
else:
    raise SystemExit('No GPU. Set Accelerator -> GPU in the settings panel. '
                     'DWN training cannot run on CPU.')

print()
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout)
print('If the nvcc CUDA version and the torch CUDA version differ a lot, the extension')
print('build in the next cell is where it will show up.')

In [ ]:
# ---- clone upstream DWN at the pinned commit and build the CUDA extension ----
# Pin matches third_party/DWN in the repo. Do not float this to main: the exporter is
# built against whatever checkpoint format this commit produces (CLAUDE.md).
PINNED_COMMIT = '9f887a0b4bd84dabf6d8c9ae35368ab2a7e0e3c0'

!rm -rf /kaggle/working/DWN
!git clone --quiet https://github.com/alanbacellar/DWN.git /kaggle/working/DWN
!cd /kaggle/working/DWN && git checkout --quiet {PINNED_COMMIT} && git log -1 --format='pinned at %h %ad %s'

# Confirm the CUDA sources actually exist at this pin before spending 5 minutes on a build.
# The pinned commit is literally "Delete custom_operators/cuda directory (Duplicate)" -- it
# removed a duplicate copy, not the real one, but that is worth verifying rather than assuming.
!ls -la /kaggle/working/DWN/src/torch_dwn/custom_operators/cuda/

# This compiles efd_cuda_kernel.cu with nvcc. Expect 2-5 minutes. It is the slowest and
# most fragile step in the notebook.
#
# --no-build-isolation is REQUIRED, not an optimization. Upstream's pyproject.toml declares
#     [build-system] requires = ["setuptools>=42", "wheel", "torch"]
# so a plain `pip install .` builds in a fresh isolated env and downloads ANOTHER torch from
# PyPI. setup.py's `import torch` then resolves to that one instead of the session's, so the
# extension gets built against a torch/CUDA pair that does not match the runtime -- it either
# fails to compile outright or builds and then fails to import on an ABI mismatch.
#
# Full output on purpose. Do NOT pipe this through `tail`: real compiler errors appear near
# the TOP of the log, while the last 20 lines are always the same generic pip epilogue
# ("did not run successfully / See above for output"), which identifies nothing.
!cd /kaggle/working/DWN && pip install --no-build-isolation .


In [ ]:
# ---- VERIFY the extension actually built ----
# This cell exists because the failure is otherwise silent. lut_layer.py does
#     if torch.cuda.is_available(): import efd_cuda
# so a failed build produces no error at install time -- you would instead get a bare
# NameError at the first forward pass, long after the real cause.
import torch, torch_dwn as dwn

try:
    import efd_cuda
    print('efd_cuda imported OK')
except ImportError as e:
    raise SystemExit(
        'efd_cuda failed to import -- the CUDA extension did not build.\n'
        'Scroll to the TOP of the install cell output and read the first compiler error;\n'
        'the tail of a pip failure is generic boilerplate and never names the cause.\n'
        f'Original error: {e}'
    )

# tiny end-to-end forward+backward, so we find out here rather than 200 lines later
_probe = torch.nn.Sequential(dwn.LUTLayer(12, 6, n=6), dwn.GroupSum(k=2, tau=1.0)).cuda()
_x = (torch.rand(4, 12, device='cuda') > 0.5).float()
_out = _probe(_x)
_out.sum().backward()
print('forward + backward OK, output shape', tuple(_out.shape))
del _probe, _x, _out


In [ ]:
# ---- the grid: the same models, at a tau that is not 10x hot ----
# Every config here already exists as a TRAINED checkpoint at tau = 3.3333. Only tau changes, so
# each row is a paired comparison against a number that has already been measured.
#
# `logit_range` and `ce_floor` are printed for reading the results; save_config() copies a fixed
# key list into the checkpoint, so they do not leak into it.
TRAINING_SET = [
 {
  "slug": "mnist_n6_z3_distributive_w1000x100_tau0p897",
  "label": "2x[1000, 100] tau=0.897",
  "group": "A-reduction",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   1000,
   100
  ],
  "mapping": [
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.8971782679756387,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 11.15,
  "ce_floor": 0.0001,
  "note": "the same config trained at 3.3333; only tau differs"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x100_tau0p897",
  "label": "2x[2000, 100] tau=0.897",
  "group": "A-reduction",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   100
  ],
  "mapping": [
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.8971782679756387,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 11.15,
  "ce_floor": 0.0001,
  "note": "the same config trained at 3.3333; only tau differs"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x300x100_tau0p897",
  "label": "3x[2000, 300, 100] tau=0.897",
  "group": "A-reduction",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   300,
   100
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.8971782679756387,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 11.15,
  "ce_floor": 0.0001,
  "note": "the same config trained at 3.3333; only tau differs"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x500x200x100_tau0p897",
  "label": "4x[2000, 500, 200, 100] tau=0.897",
  "group": "A-reduction",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   500,
   200,
   100
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.8971782679756387,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 11.15,
  "ce_floor": 0.0001,
  "note": "the same config trained at 3.3333; only tau differs"
 },
 {
  "slug": "mnist_n6_z3_distributive_w1000x500x100_tau0p897",
  "label": "3x[1000, 500, 100] tau=0.897",
  "group": "A-reduction",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   1000,
   500,
   100
  ],
  "mapping": [
   "learnable",
   "random",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.8971782679756387,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 11.15,
  "ce_floor": 0.0001,
  "note": "the same config trained at 3.3333; only tau differs"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x300x200_tau1p332",
  "label": "3x[2000, 300, 200] tau=1.332",
  "group": "A-reduction",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   300,
   200
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 1.3318822858659811,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 20,
  "logit_range": 15.02,
  "ce_floor": 0.0,
  "note": "the same config trained at 3.3333; only tau differs"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x300x500_tau2p245",
  "label": "3x[2000, 300, 500] tau=2.245",
  "group": "A-reduction",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   300,
   500
  ],
  "mapping": [
   "learnable",
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 2.2453892947761505,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 50,
  "logit_range": 22.27,
  "ce_floor": 0.0,
  "note": "the same config trained at 3.3333; only tau differs"
 },
 {
  "slug": "mnist_n6_z3_distributive_w100_tau0p897",
  "label": "1x[100] tau=0.897",
  "group": "B-ladder",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   100
  ],
  "mapping": [
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.8971782679756387,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 11.15,
  "ce_floor": 0.0001,
  "note": "same-group baseline for the tapers; the trained rung used 3.3333"
 },
 {
  "slug": "mnist_n6_z3_distributive_w200_tau1p332",
  "label": "1x[200] tau=1.332",
  "group": "B-ladder",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   200
  ],
  "mapping": [
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 1.3318822858659811,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 20,
  "logit_range": 15.02,
  "ce_floor": 0.0,
  "note": "same-group baseline for the tapers; the trained rung used 3.3333"
 },
 {
  "slug": "mnist_n6_z3_distributive_w300_tau1p678",
  "label": "1x[300] tau=1.678",
  "group": "B-ladder",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   300
  ],
  "mapping": [
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 1.6781773703074636,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 30,
  "logit_range": 17.88,
  "ce_floor": 0.0,
  "note": "same-group baseline for the tapers; the trained rung used 3.3333"
 },
 {
  "slug": "mnist_n6_z3_distributive_w500_tau2p245",
  "label": "1x[500] tau=2.245",
  "group": "B-ladder",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   500
  ],
  "mapping": [
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 2.2453892947761505,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 50,
  "logit_range": 22.27,
  "ce_floor": 0.0,
  "note": "same-group baseline for the tapers; the trained rung used 3.3333"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000_tau4p948",
  "label": "1x[2000] tau=4.948",
  "group": "B-ladder",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000
  ],
  "mapping": [
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 4.948411902096831,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 200,
  "logit_range": 40.42,
  "ce_floor": 0.0,
  "note": "same-group baseline for the tapers; the trained rung used 3.3333"
 },
 {
  "slug": "mnist_n6_z3_distributive_w100_tau0p333",
  "label": "1x[100] tau=0.333",
  "group": "C-schedule",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   100
  ],
  "mapping": [
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.33333333333333337,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 30.0,
  "ce_floor": 0.0,
  "note": "flat-range tau: 3 points on one architecture, so the schedule is measured"
 },
 {
  "slug": "mnist_n6_z3_distributive_w2000x100_tau0p333",
  "label": "2x[2000, 100] tau=0.333",
  "group": "C-schedule",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   2000,
   100
  ],
  "mapping": [
   "learnable",
   "learnable"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 0.33333333333333337,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 10,
  "logit_range": 30.0,
  "ce_floor": 0.0,
  "note": "flat-range tau: 3 points on one architecture, so the schedule is measured"
 },
 {
  "slug": "mnist_n6_z3_distributive_w1000x500_tau2p245",
  "label": "2x[1000, 500] tau=2.245",
  "group": "B-ladder",
  "n": 6,
  "thermometer_bits": 3,
  "layers": [
   1000,
   500
  ],
  "mapping": [
   "learnable",
   "random"
  ],
  "thermometer": "distributive",
  "num_classes": 10,
  "tau": 2.2453892947761505,
  "batch_size": 100,
  "epochs": 30,
  "lr": 0.01,
  "lr_step": 14,
  "lr_gamma": 0.1,
  "seed": 20260811,
  "groupsum_group": 50,
  "logit_range": 22.27,
  "ce_floor": 0.0,
  "note": "THE PAPER'S CONFIG, and the one the tau correction missed. Its FINAL layer is 500, so its group is 50 and it wanted 2.245 -- the same ~48% error 1x[500] had. It was skipped because the correction was scoped as a ladder fix and this sits in the multilayer group. Trained at 3.3333; expect roughly +0.27 pp, which is at the 0.24 pp noise floor"
 }
]

# The tau every one of these was FIRST trained at, and which this notebook exists to correct.
OLD_TAU = 3.3333333333333335

ONLY_SLUGS = None       # e.g. one slug to sanity-check the schedule before committing a session
ONLY_N = None           # cap configs per session; None runs until the grid is done
PRECOMPUTE_LIMIT_GB = 6.0

print(f'{len(TRAINING_SET)} configs')
for g in sorted({c['group'] for c in TRAINING_SET}):
    print(f"  {g:12s} {sum(1 for c in TRAINING_SET if c['group'] == g)}")
print()
print(f'{"config":34s} {"group":>6} {"tau":>8} {"range":>7} {"CE floor":>9}   was')
print('-' * 82)
for c in TRAINING_SET:
    old_range = c['groupsum_group'] / OLD_TAU
    old_floor = __import__('math').log(1 + 9 * __import__('math').exp(-old_range))
    print(f"  {'x'.join(str(x) for x in c['layers']):32s} {c['groupsum_group']:>6d} "
          f"{c['tau']:>8.3f} {c['logit_range']:>7.2f} {c['ce_floor']:>9.4f}   "
          f"range {old_range:.2f}, floor {old_floor:.4f}")


In [ ]:
# ---- load MNIST ----
# Matches third_party/DWN/examples/mnist.py: the canonical 60k/10k split, pixels in [0, 1].
# NO StandardScaler -- upstream uses transforms.ToTensor(), which is exactly /255. Our exporter
# never reads ck['scaler'], but the key is written below anyway so the checkpoint keeps the same
# shape as the JSC ones (mean 0, scale 255 reproduces this transform exactly).
import numpy as np, torch
from sklearn.datasets import fetch_openml

SEED = TRAINING_SET[0]['seed']
torch.manual_seed(SEED)
np.random.seed(SEED)

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_all = mnist.data.astype(np.float32) / 255.0        # [0, 1], the ToTensor transform
y_all = mnist.target.astype(np.int64)
assert X_all.shape[1] == 784 and len(np.unique(y_all)) == 10

# The canonical split -- mnist_784 is ordered train-then-test, so this is the standard one every
# published MNIST number uses. Do not shuffle before slicing.
X_train, X_test = X_all[:60000], X_all[60000:]
y_train, y_test = y_all[:60000], y_all[60000:]

N_FEATURES = X_train.shape[1]
X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
y_train_t = torch.from_numpy(y_train).long()
y_test_t = torch.from_numpy(y_test).long()
print('train', tuple(X_train_t.shape), ' test', tuple(X_test_t.shape))
print('pixel range', float(X_train_t.min()), '-', float(X_train_t.max()))
print(f'{(X_train_t == 0).float().mean():.1%} of training pixels are exactly zero -- so many '
      'quantile thresholds will coincide, and duplicate comparators cost nothing in hardware.')


In [ ]:
# ---- helpers ----
import time, os, gc
import torch_dwn as dwn
from torch import nn
from torch.nn.functional import cross_entropy

THERMOMETERS = {
    'plain': dwn.Thermometer,
    'linear': dwn.Thermometer,            # grid calls evenly-spaced 'linear'
    'gaussian': dwn.GaussianThermometer,
    'distributive': dwn.DistributiveThermometer,
}
WORK = '/kaggle/working'


def binarize_chunked(therm, x, chunk=10000):
    """Binarize -> flatten -> uint8 in slices, to bound peak memory (Phase 1 cell 6)."""
    out = torch.empty((x.size(0), x.size(1) * therm.num_bits), dtype=torch.uint8)
    for i in range(0, x.size(0), chunk):
        out[i:i + chunk] = therm.binarize(x[i:i + chunk]).flatten(start_dim=1).to(torch.uint8)
    return out


def make_group(kind, z):
    """Fit the thermometer and prepare batch access for every config sharing (kind, z)."""
    therm = THERMOMETERS[kind](z).fit(X_train_t)
    gb = (X_train_t.size(0) + X_test_t.size(0)) * N_FEATURES * z / 1e9
    thr_gpu = therm.thresholds.cuda()

    def on_the_fly(xf):
        # Same comparison binarization.py performs, done on the GPU so no huge CPU tensor
        # is ever materialized.
        return (xf.cuda().unsqueeze(-1) > thr_gpu).flatten(start_dim=1).float()

    if gb <= PRECOMPUTE_LIMIT_GB:
        xb_tr, xb_te = binarize_chunked(therm, X_train_t), binarize_chunked(therm, X_test_t)
        # The two paths MUST agree, or a large-z config would be trained on different bits
        # than a small-z one -- a difference that would look like a result.
        probe = on_the_fly(X_test_t[:256]).to(torch.uint8).cpu()
        assert torch.equal(probe, xb_te[:256]), 'on-the-fly binarization != precomputed'
        print(f'    precomputed {gb:.2f} GB uint8 (GPU path verified identical on 256 samples)')
        return therm, (lambda i: xb_tr[i].cuda().float()), (lambda i: xb_te[i].cuda().float()), \
            xb_tr.size(1), (xb_tr, xb_te)

    print(f'    on-the-fly binarization ({gb:.2f} GB would not fit)')
    return therm, (lambda i: on_the_fly(X_train_t[i])), (lambda i: on_the_fly(X_test_t[i])), \
        N_FEATURES * z, None


def train_one(cfg, get_train, get_test, in_bits):
    """Train one config. Returns (model, results dict)."""
    torch.manual_seed(cfg['seed'])
    layers, size = [], in_bits
    for i, w in enumerate(cfg['layers']):
        layers.append(dwn.LUTLayer(size, w, n=cfg['n'], mapping=cfg['mapping'][i]))
        size = w
    layers.append(dwn.GroupSum(k=cfg['num_classes'], tau=cfg['tau']))
    model = nn.Sequential(*layers).cuda()

    opt = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
    sched = torch.optim.lr_scheduler.StepLR(opt, cfg['lr_step'], cfg['lr_gamma'])
    n_train, n_test = X_train_t.size(0), X_test_t.size(0)

    def evaluate(chunk=5000):
        model.eval()
        correct = 0
        with torch.no_grad():
            for i in range(0, n_test, chunk):
                sl = slice(i, i + chunk)
                correct += (model(get_test(sl)).argmax(1)
                            == y_test_t[sl].cuda()).sum().item()
        return correct / n_test

    best, history, losses = 0.0, [], []
    t0 = time.time()
    for ep in range(cfg['epochs']):
        model.train()
        perm = torch.randperm(n_train)
        run, nb = 0.0, 0
        for i in range(0, n_train, cfg['batch_size']):
            idx = perm[i:i + cfg['batch_size']]
            opt.zero_grad()
            loss = cross_entropy(model(get_train(idx)), y_train_t[idx].cuda())
            loss.backward()
            opt.step()
            run += loss.item(); nb += 1
        sched.step()
        losses.append(run / nb)
        acc = evaluate()
        history.append(acc); best = max(best, acc)
        if ep % 8 == 7 or ep == cfg['epochs'] - 1:
            print(f'      epoch {ep+1:3d}/{cfg["epochs"]}  loss {losses[-1]:.4f}  '
                  f'acc {acc:.4f}  [{time.time()-t0:.0f}s]')
    return model, {'final_acc': history[-1], 'best_acc': best,
                   'history': history, 'epoch_losses': losses,
                   'seconds': round(time.time() - t0, 1)}


def save_config(cfg, model, therm, res, get_test):
    """Checkpoint + 1000-sample test vectors, in the format exporter/extract.py reads."""
    slug = cfg['slug']
    ck_path = f'{WORK}/{slug}_checkpoint.pt'
    torch.save({
        'run_name': slug,
        # Exactly the keys the Phase 1 checkpoint carries -- extract.py reads config['n'],
        # ['layers'], ['thermometer_bits'], ['num_classes'] (docs/checkpoint-format.md).
        'config': {k: cfg[k] for k in (
            'thermometer', 'thermometer_bits', 'n', 'layers', 'mapping', 'num_classes',
            'tau', 'batch_size', 'epochs', 'lr', 'lr_step', 'lr_gamma', 'seed')},
        'pinned_commit': PINNED_COMMIT,
        'state_dict': model.state_dict(),
        'thermometer': {'kind': cfg['thermometer'], 'num_bits': cfg['thermometer_bits'],
                        'thresholds': therm.thresholds.cpu()},
        # x_scaled = (x_raw - 0) / 255, i.e. exactly what ToTensor does. Same shape as the
        # JSC checkpoints' StandardScaler entry so nothing downstream has to special-case it.
        'scaler': {'mean': torch.zeros(N_FEATURES),
                   'scale': torch.full((N_FEATURES,), 255.0)},
        'classes': [str(i) for i in range(10)],
        'feature_names': [f'pixel{i}' for i in range(N_FEATURES)],
        'results': res,
        'grid_label': cfg['label'],
        'torch_version': torch.__version__,
    }, ck_path)

    # Gate 1 needs these: gen_vectors.py reads <run>_testvectors.npz beside the checkpoint.
    # x_binarized drives the core testbench, x_raw the encoder+core one -- both, so a Gate 1
    # failure localizes to one or the other.
    model.eval()
    with torch.no_grad():
        xb = get_test(slice(0, 1000))
        pred = model(xb).argmax(1).cpu().numpy()
    np.savez_compressed(
        f'{WORK}/{slug}_testvectors.npz',
        x_binarized=xb.to(torch.uint8).cpu().numpy(),
        x_raw=X_test[:1000], y=y_test[:1000], pred=pred)
    return ck_path

In [ ]:
# ---- carry forward anything already trained in an EARLIER SESSION ----
# /kaggle/working is fresh on every version -- it only persists while one session is alive.
# So the resume check below would restart from zero on a new session, which is precisely the
# case a 32-config run needs. Add the previous run's OUTPUT as an input dataset and this copies
# it forward, so the final Output panel holds the complete set rather than one session's slice.
import glob, shutil

carried = 0
for src in glob.glob('/kaggle/input/**/*_checkpoint.pt', recursive=True) + \
           glob.glob('/kaggle/input/**/*_testvectors.npz', recursive=True):
    dst = os.path.join(WORK, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        carried += 1
print(f'carried forward {carried} file(s) from /kaggle/input')
if not carried:
    print('(none -- first session, or no previous output added as an input dataset)')

In [ ]:
# ---- train the grid ----
# Ordered by (encoding, z) so each binarization is built once and reused by every config that
# shares it. Within a group, configs run cheapest-first so a session that dies late still
# banks the most models.
import collections

# Filter FIRST. The training loop below iterates by_group, not TRAINING_SET, so restricting
# TRAINING_SET after by_group is built changes only the printed counts and trains the whole grid
# anyway -- lowest z first, which is not the config anyone asked for.
if ONLY_SLUGS:
    TRAINING_SET = [c for c in TRAINING_SET if c['slug'] in ONLY_SLUGS]
    assert TRAINING_SET, 'ONLY_SLUGS matched nothing -- check the slug spelling'
    print(f'ONLY_SLUGS: restricted to {len(TRAINING_SET)} config(s): '
          + ', '.join(c['slug'] for c in TRAINING_SET))

by_group = collections.defaultdict(list)
for c in TRAINING_SET:
    by_group[(c['thermometer'], c['thermometer_bits'])].append(c)
for v in by_group.values():
    v.sort(key=lambda c: sum(c['layers']))

done = [c for c in TRAINING_SET if os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
todo = [c for c in TRAINING_SET if c not in done]
print(f'{len(done)} already trained, {len(todo)} to go')
if ONLY_N:
    print(f'ONLY_N={ONLY_N}: stopping after {ONLY_N} this session')

summary, trained = [], 0
for (kind, z), configs in sorted(by_group.items(), key=lambda kv: kv[0][1]):
    pending = [c for c in configs
               if not os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
    if not pending or (ONLY_N and trained >= ONLY_N):
        continue
    print(f'\n=== {kind}, z={z} -- {len(pending)} config(s) ===')
    therm, get_train, get_test, in_bits, held = make_group(kind, z)

    for cfg in pending:
        if ONLY_N and trained >= ONLY_N:
            break
        print(f'  -- {cfg["slug"]}  ({cfg["label"]}, {sum(cfg["layers"])} nodes)')
        model, res = train_one(cfg, get_train, get_test, in_bits)
        path = save_config(cfg, model, therm, res, get_test)
        print(f'     final {res["final_acc"]:.4f}  best {res["best_acc"]:.4f}  '
              f'{res["seconds"]:.0f}s  -> {os.path.basename(path)}')
        summary.append((cfg['label'], cfg['slug'], res['final_acc'], res['seconds']))
        trained += 1
        del model; gc.collect(); torch.cuda.empty_cache()

    del therm, get_train, get_test, held
    gc.collect(); torch.cuda.empty_cache()

print(f'\ntrained {trained} config(s) this session')

In [ ]:
# ---- summary ----
import glob
cks = sorted(glob.glob(f'{WORK}/*_checkpoint.pt'))
print(f'{len(cks)} checkpoints in {WORK} ({len(TRAINING_SET)} configs in the grid)')
print()
if summary:
    print(f'{"config":24s} {"final acc":>10} {"seconds":>9}')
    print('-' * 46)
    for label, slug, acc, secs in summary:
        print(f'{label:24s} {100*acc:>9.2f}% {secs:>9.0f}')
    print('-' * 46)

missing = [c['slug'] for c in TRAINING_SET
           if not os.path.exists(f"{WORK}/{c['slug']}_checkpoint.pt")]
if missing:
    print(f'\nSTILL MISSING ({len(missing)}): re-run this notebook to continue.')
    for s in missing[:10]:
        print('  ', s)
    if len(missing) > 10:
        print(f'   ... and {len(missing)-10} more')
else:
    print('\nAll configs trained. Download every *_checkpoint.pt AND *_testvectors.npz')
    print('into training/artifacts/, then run:  python dse/run.py --all --impl')

## After this runs

Download **both** files per config into `training/artifacts/mnist/`. No new RTL is needed —
`rtlgen/emit_core.py` already loops over layers, and `tau` is a training-time constant that never
reaches hardware.

```
python scripts/run_gate1.py --checkpoint training/artifacts/mnist/<slug>_checkpoint.pt
python scripts/run_synth.py --rtl-dir build/rtl
```

**Gate 1 first, always.** Area for unverified RTL describes nothing.

## Reading the result

Three questions, in the order they have to be answered.

**1. Was it tau?** Compare each A row against its τ=3.3333 twin. If the group-10 tapers move from
~91% up toward the ladder, the previous notebook measured the schedule and its architectural
conclusion is void. If they *don't* move, tau was not the constraint and a 10-bit floor genuinely
cannot carry MNIST — which is itself a clean result, and a stronger one for having been tested.

**2. Does a learned taper beat a plain narrow layer?** Only now is this comparison honest, with
both sides at the same group and the same tau:

| taper | vs same-group single layer |
|---|---|
| `2x[2000, 100]` | `1x100` |
| `3x[2000, 300, 200]` | `1x200` |
| `3x[2000, 300, 500]` | `1x500` |

At τ=3.3333 those gaps were **+3.09 / +2.43 / +0.13 pp** — the taper doing real work at narrow
outputs and nothing at all by 500. Whether that survives the correction is the actual finding.

**3. Which schedule?** Three points on `1x100` (τ = 3.333, 0.897, 0.333) and two on
`2x[2000,100]`. If flat-range wins, the logit range is what matters and MNIST's schedule is not
JSC's. If power-law wins, the exponent transfers across datasets — which would be worth saying
out loud, because nobody has checked.

⚠️ **Differences below the noise floor are not differences.** JSC measured run-to-run spread at
0.15 pp. **No equivalent figure exists for MNIST** — the cheapest way to get one is to retrain a
single config under a second seed, and until that is done, treat gaps under ~0.3 pp as unresolved
rather than small.

## What is still not answered by any of this

Area. Every number in both notebooks is accuracy; the reason to want a taper at all is that the
popcount was **35% of the JSC core and on the critical path**. `3x[2000, 300, 500]` already
trades a 4× smaller adder tree for 800 extra nodes at −0.61 pp, and whether that is a win is a
synthesis question, not a training one.
